# 11 · SAR

Template para el modelo autoregresivo espacial implementado en `SpatialAutoregressiveModel`.

## Hipótesis del modelo

- Parte del precio se explica por dependencia espacial entre propiedades cercanas.
- El parámetro espacial `rho` debería capturar interacción local no absorbida por las covariables.
- La interpretación principal viene por coeficientes globales, `rho` y patrones de residuos.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from ml_core.models.sarModel import SpatialAutoregressiveModel

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "output" / "11_sar"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

## Datos y split

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "rental_listings_model_input.parquet"
# df = pd.read_parquet(DATA_PATH)

target_col = "price_m2"
coord_cols = ["lon", "lat"]
feature_cols = []

# Reutilizar el split definido en 01_baseline_and_protocol.

## Entrenamiento

In [ ]:
sar_config = {"k": 8}

# X_train = train_df[feature_cols]
# y_train = train_df[target_col]
# coords_train = train_df[coord_cols].to_numpy()
# model = SpatialAutoregressiveModel(k=sar_config["k"])
# model.fit(X_train, y_train, coords_train)
# print(model.summary())

## Sensibilidad de hiperparámetros

Acá conviene comparar distintos valores de `k` y, si querés, distintas definiciones de vecindad.

In [ ]:
# candidate_k = [4, 6, 8, 10, 15]
# tuning_rows = []
# for k in candidate_k:
#     tuned = SpatialAutoregressiveModel(k=k)
#     tuned.fit(X_train, y_train, coords_train)
#     pred_valid = tuned.predict(valid_df[feature_cols], valid_df[coord_cols].to_numpy())
#     tuning_rows.append({"k": k, "rmse_valid": float(np.sqrt(np.mean((valid_df[target_col] - np.asarray(pred_valid).reshape(-1)) ** 2)))})
# pd.DataFrame(tuning_rows).sort_values("rmse_valid")

## Evaluación global

In [ ]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    residuals = y_true - y_pred
    return {
        "rmse": float(np.sqrt(np.mean(residuals ** 2))),
        "mae": float(np.mean(np.abs(residuals))),
        "r2": float(1 - np.sum(residuals ** 2) / np.sum((y_true - y_true.mean()) ** 2)),
        "bias": float(residuals.mean()),
    }

# X_test = test_df[feature_cols]
# coords_test = test_df[coord_cols].to_numpy()
# y_test = test_df[target_col]
# y_pred = model.predict(X_test, coords_test)
# metrics = regression_metrics(y_test, y_pred)
# metrics

## Interpretación

A diferencia de GWR, SAR tiene una interpretación más global:

- coeficientes globales
- magnitud y signo de `rho`
- sensibilidad al número de vecinos
- concentración espacial de residuos

In [ ]:
# coef_frame = pd.DataFrame({
#     "feature": ["intercept"] + feature_cols + ["rho"],
#     "coefficient": np.asarray(model.model_.betas).reshape(-1),
# })
# coef_frame

## Residuos y export

In [ ]:
# test_export = test_df[[target_col] + coord_cols].copy()
# test_export = test_export.rename(columns={target_col: "y_true"})
# test_export["y_pred"] = np.asarray(y_pred).reshape(-1)
# test_export["residual"] = test_export["y_true"] - test_export["y_pred"]
# test_export["split"] = "test"
# test_export.to_parquet(OUTPUT_DIR / "test_predictions.parquet", index=False)
# coef_frame.to_parquet(OUTPUT_DIR / "interpretability.parquet", index=False)
# (OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False))
# (OUTPUT_DIR / "run_config.json").write_text(json.dumps(sar_config, indent=2, ensure_ascii=False))